# PubMed Document Retrieval & Cleaning Pipeline
Fetches full abstracts from PubMed URLs in BioASQ snippets instead of using snippet text.

In [1]:
# ── Cell 1: Imports + Config ────────────────────────────────────────
import os
import json
import time
import requests
import unicodedata
import re
import ftfy
from tqdm import tqdm
from langdetect import detect, LangDetectException, DetectorFactory
from dotenv import load_dotenv

load_dotenv()
DetectorFactory.seed = 0

# NCBI API key (free at https://www.ncbi.nlm.nih.gov/account/)

NCBI_API_KEY = '0c02b6e7f8e56927b51f9421816475b02e08'
RATE_LIMIT = 0.11 if NCBI_API_KEY else 0.34  # 10/sec with key, 3/sec without

DATA_DIR = os.environ.get('DATA_DIR', '/workspace/data')

print(f'NCBI API key: {"✅ found" if NCBI_API_KEY else "❌ not set (rate limited to 3 req/sec)"}')
print(f'Rate limit delay: {RATE_LIMIT}s per request')
print(f'Data dir: {DATA_DIR}')

NCBI API key: ✅ found
Rate limit delay: 0.11s per request
Data dir: /workspace/data


In [2]:
# ── Cell 2: Load Categorized Training Data ──────────────────────────
input_path = os.path.join(DATA_DIR, 'training_categorized.json')

with open(input_path, 'r', encoding='utf-8') as f:
    training_data = json.load(f)

print(f'✅ Loaded {len(training_data)} docs from {input_path}')
print(f'Fields: {list(training_data[0].keys())}')

# Preview snippet URLs from first doc
sample = training_data[0]
print(f'\nSample question: {sample.get("body", "")[:100]}')
print(f'Snippets: {len(sample.get("snippets", []))}')
for s in sample.get('snippets', [])[:3]:
    print(f'  URL:  {s.get("document", "")}')
    print(f'  Text: {s.get("text", "")[:80]}...')

✅ Loaded 4803 docs from /workspace/data/training_categorized.json
Fields: ['body', 'documents', 'ideal_answer', 'concepts', 'type', 'id', 'snippets', 'text', 'category', 'topic_id']

Sample question: Is Hirschsprung disease a mendelian or a multifactorial disorder?
Snippets: 16
  URL:  http://www.ncbi.nlm.nih.gov/pubmed/15829955
  Text: Hirschsprung disease (HSCR) is a multifactorial, non-mendelian disorder in which...
  URL:  http://www.ncbi.nlm.nih.gov/pubmed/15617541
  Text: In this study, we review the identification of genes and loci involved in the no...
  URL:  http://www.ncbi.nlm.nih.gov/pubmed/12239580
  Text: Coding sequence mutations in e.g. RET, GDNF, EDNRB, EDN3, and SOX10 lead to long...


In [3]:
# ── Cell 3: PubMed Fetch Function ───────────────────────────────────
def extract_pmid(url: str) -> str:
    """Extract PMID from PubMed URL."""
    # e.g. http://www.ncbi.nlm.nih.gov/pubmed/35410609 → 35410609
    return url.rstrip('/').split('/')[-1]

def fetch_pubmed_abstract(url: str, api_key: str = None) -> str:
    """Fetch full abstract text from a PubMed URL."""
    try:
        pmid = extract_pmid(url)
        if not pmid.isdigit():
            return ""

        params = {
            "db": "pubmed",
            "id": pmid,
            "rettype": "abstract",
            "retmode": "text"
        }
        if api_key:
            params["api_key"] = api_key

        response = requests.get(
            "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi",
            params=params,
            timeout=10
        )
        if response.status_code == 200:
            return response.text.strip()
        return ""
    except Exception as e:
        return ""

# Test with one URL
test_url = training_data[0].get('snippets', [{}])[0].get('document', '')
print(f'Testing URL: {test_url}')
abstract = fetch_pubmed_abstract(test_url, NCBI_API_KEY)
print(f'\nFetched abstract ({len(abstract)} chars):')
print(abstract[:500])

Testing URL: http://www.ncbi.nlm.nih.gov/pubmed/15829955

Fetched abstract (1639 chars):
1. Nature. 2005 Apr 14;434(7035):857-63. doi: 10.1038/nature03467.

A common sex-dependent mutation in a RET enhancer underlies Hirschsprung disease 
risk.

Emison ES(1), McCallion AS, Kashuk CS, Bush RT, Grice E, Lin S, Portnoy ME, 
Cutler DJ, Green ED, Chakravarti A.

Author information:
(1)McKusick-Nathans Institute of Genetic Medicine, Johns Hopkins University 
School of Medicine, Baltimore, Maryland 21205, USA.

The identification of common variants that contribute to the genesis of human 



In [4]:
# ── Cell 4: Text Cleaning Functions ─────────────────────────────────
def remove_html_tags(text: str) -> str:
    return re.sub(r'<[^>]+>', '', text)

def fix_encoding(text: str) -> str:
    return ftfy.fix_text(text)

def normalize_text(text: str, form: str = 'NFC') -> str:
    text = unicodedata.normalize(form, text)
    text = re.sub(r'\r\n|\r', '\n', text)
    text = re.sub(r'\t', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    return text.strip()

def is_low_quality(text: str, min_words: int = 5, max_symbol_ratio: float = 0.3) -> bool:
    words = text.split()
    if len(words) < min_words:
        return True
    symbols = sum(1 for c in text if not c.isalnum() and not c.isspace())
    if symbols / max(len(text), 1) > max_symbol_ratio:
        return True
    return False

def detect_language(text: str, target_lang: str = 'en') -> bool:
    try:
        return detect(text) == target_lang
    except LangDetectException:
        return False

def remove_pii(text: str) -> str:
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.]+', '[EMAIL]', text)
    text = re.sub(r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b', '[PHONE]', text)
    text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[SSN]', text)
    return text

def clean_text(text: str) -> str | None:
    """Full cleaning pipeline for fetched abstract text."""
    text = remove_html_tags(text)
    text = fix_encoding(text)
    text = normalize_text(text)
    if is_low_quality(text):
        return None
    if not detect_language(text):
        return None
    text = remove_pii(text)
    return text

print('✅ Cleaning functions ready')

✅ Cleaning functions ready


In [5]:
# ── Cell 5: Deduplicate URLs (avoid redundant fetches) ───────────────
# Collect all unique PubMed URLs across all docs
all_urls = set()
for doc in training_data:
    for snippet in doc.get('snippets', []):
        url = snippet.get('document', '')
        if url:
            all_urls.add(url)

print(f'Total docs:        {len(training_data)}')
print(f'Unique PubMed URLs: {len(all_urls)}')

# Estimate time
est_seconds = len(all_urls) * RATE_LIMIT
est_minutes = est_seconds / 60
print(f'\nEstimated fetch time: {est_minutes:.1f} minutes')
print(f'(Based on {RATE_LIMIT}s/request × {len(all_urls)} unique URLs)')

Total docs:        4803
Unique PubMed URLs: 37596

Estimated fetch time: 68.9 minutes
(Based on 0.11s/request × 37596 unique URLs)


In [6]:
# ── Cell 6: Fetch All Unique Abstracts ───────────────────────────────
# Cache results so each URL is only fetched once
abstract_cache = {}
failed_urls = []

for url in tqdm(all_urls, desc='Fetching PubMed abstracts'):
    abstract = fetch_pubmed_abstract(url, NCBI_API_KEY)
    if abstract:
        abstract_cache[url] = abstract
    else:
        failed_urls.append(url)
    time.sleep(RATE_LIMIT)  # respect rate limit

print(f'\n✅ Fetched:  {len(abstract_cache)} abstracts')
print(f'❌ Failed:   {len(failed_urls)} URLs')

# Save cache to avoid re-fetching
cache_path = os.path.join(DATA_DIR, 'pubmed_abstract_cache.json')
with open(cache_path, 'w', encoding='utf-8') as f:
    json.dump(abstract_cache, f, indent=2, ensure_ascii=False)
print(f'✅ Cache saved to {cache_path}')

Fetching PubMed abstracts:   0% 109/37596 [01:54<10:54:55,  1.05s/it]


KeyboardInterrupt: 

In [ ]:
# ── Cell 7: Load Cache (skip re-fetching on reruns) ──────────────────
cache_path = os.path.join(DATA_DIR, 'pubmed_abstract_cache.json')

if os.path.exists(cache_path):
    with open(cache_path, 'r', encoding='utf-8') as f:
        abstract_cache = json.load(f)
    print(f'✅ Loaded {len(abstract_cache)} cached abstracts')
else:
    print('❌ No cache found — run Cell 6 first')

In [ ]:
# ── Cell 8: Build Enriched Documents ────────────────────────────────
enriched_docs = []
skipped = 0

for doc in tqdm(training_data, desc='Building enriched docs'):
    question = doc.get('body', '')
    snippets = doc.get('snippets', [])

    # Collect full abstract texts for each snippet URL
    full_texts = []
    source_urls = []

    for snippet in snippets:
        url = snippet.get('document', '')
        if url in abstract_cache:
            cleaned = clean_text(abstract_cache[url])
            if cleaned:
                full_texts.append(cleaned)
                source_urls.append(url)
        else:
            # Fallback to snippet text if URL not cached
            fallback = clean_text(snippet.get('text', ''))
            if fallback:
                full_texts.append(fallback)

    if not full_texts:
        skipped += 1
        continue

    enriched_docs.append({
        **doc,
        "text": " ".join(full_texts),  # full abstracts joined
        "question": question,           # original question
        "source_urls": source_urls,     # PubMed URLs for traceability
        "n_sources": len(source_urls)
    })

print(f'\n✅ Enriched docs: {len(enriched_docs)}')
print(f'⚠️  Skipped:       {skipped} (no usable text found)')

# Preview
sample = enriched_docs[0]
print(f'\nSample question: {sample["question"][:100]}')
print(f'Sources: {sample["n_sources"]}')
print(f'Text ({len(sample["text"])} chars): {sample["text"][:300]}...')

In [ ]:
# ── Cell 9: Save Enriched Documents ─────────────────────────────────
output_path = os.path.join(DATA_DIR, 'training_enriched.json')

with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(enriched_docs, f, indent=2, ensure_ascii=False)

print(f'✅ Saved {len(enriched_docs)} enriched docs to {output_path}')

# Verify
with open(output_path, 'r', encoding='utf-8') as f:
    verify = json.load(f)

print(f'✅ Verified: {len(verify)} docs loaded back')
print(f'   Fields: {list(verify[0].keys())}')
print(f'   Category: {verify[0].get("category")}')
print(f'   Sources: {verify[0].get("n_sources")}')
print(f'   Text preview: {verify[0]["text"][:150]}...')